In [1]:
import torch, sys
print('python:', sys.version.split()[0], '| torch:', torch.__version__)
print('cuda  :', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Enable GPU: Settings > Accelerator > GPU T4'


/home/fat-potato/prj/sharedGPU/.venv/lib/python3.13/site-packages/torch/_subclasses/functional_tensor.py:368: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


python: 3.13.14 | torch: 2.13.0+cu130
cuda  : False | CPU only


AssertionError: Enable GPU: Settings > Accelerator > GPU T4

## Why the cell above fails here

This machine has no GPU (`nvidia-smi` doesn't even exist) — the Docker control-plane container is pure CPU orchestration too. The only real GPU is the **Colab notebook you connected earlier** (Tesla T4, joins the pool over a WebSocket through the ngrok tunnel).

So heavy GPU work can't run *in this kernel*. Instead, this notebook acts as a **client**: it calls `PoolClient.deploy(...).infer(...)`, the control plane routes the call to the connected Colab T4 over the WebSocket, runs it there, and streams the result back here.

In [2]:
import sys, json
sys.path.insert(0, "backend_scripts")
from _common import make_client

pool = make_client()

# Pick the real Colab/Kaggle node dynamically (don't hardcode its id -- it changes on reconnect).
nodes = pool.nodes()
gpu_node = next((n for n in nodes if n["provider"] in ("colab", "kaggle")), None)
if gpu_node is None:
    raise RuntimeError("No real Colab/Kaggle GPU node connected -- start one first (see node/colab_kaggle_node.py).")

print(f"Using {gpu_node['gpuName']} on node {gpu_node['nodeId']} ({gpu_node['freeVramGib']} GiB free)")

dep = pool.deploy(
    {"kind": "python", "entrypoint": "pool_models.gpu_probe:build"},
    mem_gib=1,
    node_id=gpu_node["nodeId"],
)
try:
    result = dep.infer({"size": 4096})
    print(json.dumps(result, indent=2))
    assert result["cuda"], "Expected this to run on the real GPU"
    print(f"\n✓ Ran on real GPU: {result['device']} ({result['matmulMs']} ms for a 4096² fp16 matmul)")
finally:
    dep.teardown()


[gpu-pool] target: https://a345-113-212-110-17.ngrok-free.app
Using Tesla T4 on node node-bc43c2c6bf2b (14.56 GiB free)
{
  "cuda": true,
  "device": "Tesla T4",
  "host": "5196961026f2",
  "computeCapability": "7.5",
  "matmulSize": 4096,
  "matmulMs": 6.5,
  "checksum": 374099.28125,
  "vramFreeGib": 11.35,
  "vramTotalGib": 14.56
}

✓ Ran on real GPU: Tesla T4 (6.5 ms for a 4096² fp16 matmul)


## Swapping in your actual heavy workload

Replace the `model_spec` above with whatever you actually need to run:

**A Hugging Face model** (no new code — the node builds a `transformers` pipeline):
```python
dep = pool.deploy(
    {"kind": "transformers", "task": "text-generation", "model": "<hf-model-id>"},
    mem_gib=8,                     # size to the model's VRAM footprint
    node_id=gpu_node["nodeId"],
)
dep.infer({"inputs": "..."})
```

**Custom PyTorch code**: add a module under `pool_models/`, e.g. `pool_models/my_workload.py`:
```python
def build():
    import torch
    class Model:
        def infer(self, payload):
            ...  # your heavy GPU code, uses torch.cuda normally -- it runs ON the node
            return {"result": ...}
    return Model()
```
then `{"kind": "python", "entrypoint": "pool_models.my_workload:build"}`. Push the change to `notAvailable73/sharedGPU` first — nodes auto-pull `pool_models/` from that repo on each deploy, no notebook restart needed.

`mem_gib` just needs to roughly match what the model actually uses (VRAM the scheduler reserves) — check `gpu_node["freeVramGib"]` before deploying something large.

## Actually running a GPU-heavy task: real LLM generation on the node

The matmul probe above proves the wiring; it's not real work. This deploys `pool_models.llm:build` (🤗 transformers), which loads **Qwen2.5-1.5B-Instruct** onto the connected node's T4 in fp16 and runs text generation -- a real model load + real autoregressive decode, reporting tokens/sec and VRAM used.

In [4]:
import json

dep = pool.deploy(
    {"kind": "python", "entrypoint": "pool_models.llm:build",
     "args": {"model": "Qwen/Qwen2.5-1.5B-Instruct", "dtype": "float16"}},
    mem_gib=4,
    node_id=gpu_node["nodeId"],
)
try:
    result = dep.infer({
        "prompt": "What is the capital of dhaka",
        "max_new_tokens": 128,
    })
    print(json.dumps(result, indent=2))
    assert result["useCuda"], "Expected generation to run on the real GPU"
    print(f"\n✓ Generated {result['genTokens']} tokens in {result['genSeconds']}s "
          f"({result['tokensPerSec']} tok/s) on {result['device']}, "
          f"{result['vramUsedGib']} GiB VRAM used")
finally:
    dep.teardown()


{
  "text": "The capital of Bangladesh is Dhaka.",
  "model": "Qwen/Qwen2.5-1.5B-Instruct",
  "promptTokens": 36,
  "genTokens": 9,
  "genSeconds": 0.369,
  "tokensPerSec": 24.38,
  "useCuda": true,
  "device": "Tesla T4",
  "vramUsedGib": 3.22,
  "vramTotalGib": 14.56
}

✓ Generated 9 tokens in 0.369s (24.38 tok/s) on Tesla T4, 3.22 GiB VRAM used
